In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision import transforms, models
import timm
from IPython.display import display
import ipywidgets as widgets
import torch
import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
RESULT_BASE = '/content/drive/MyDrive/DS200_BIG_DATA'

DATASET_CONFIGS = {
    "Shenzhen": {'mean': [0.6143]*3, 'std': [0.2564]*3, 'is_rgb': True},
    "JSRT":     {'mean': [0.595]*3,  'std': [0.2743]*3, 'is_rgb': True},
    "LUNA":     {'mean': [0.8118]*3, 'std': [0.3791]*3, 'is_rgb': False},
}

MODEL_OPTIONS = {
    'UNet (Baseline)':       'unet',
    'CE-Net ResNet34':       'r34',
    'CE-Net EfficientB4':    'effb4',
    'CE-Net MobileNetV3':    'mobilev3',
    'CE-Net SwinT':          'swin',
}


LOSS_OPTIONS = ['dice']

In [ ]:
# ==========================================
# BASELINE - UNET - GITHUB - TRAIN FROM SCRATCH
# ==========================================
class DoubleConv(nn.Module):
    """Two consecutive conv + BN + ReLU"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        else:
            self.up = nn.ConvTranspose2d(in_channels // 2, in_channels // 2, kernel_size=2, stride=2)

        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        # Padding để match kích thước (nếu cần)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])

        # Concatenate
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, n_classes):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, n_classes, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, bilinear=True):
        super(UNet, self).__init__()

        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 512)

        self.up1 = Up(1024, 256, bilinear)
        self.up2 = Up(512, 128, bilinear)
        self.up3 = Up(256, 64, bilinear)
        self.up4 = Up(128, 64, bilinear)

        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)

        logits = self.outc(x)

        return logits  # Dùng cho binary segmentation

In [ ]:
# ==========================================
# CE-NET MODEL
# ==========================================
class DACblock(nn.Module):
    def __init__(self, channel):
        super().__init__()
        self.dilate1 = nn.Conv2d(channel, channel, 3, dilation=1, padding=1)
        self.dilate2 = nn.Conv2d(channel, channel, 3, dilation=3, padding=3)
        self.dilate3 = nn.Conv2d(channel, channel, 3, dilation=5, padding=5)
        self.conv1x1 = nn.Conv2d(channel, channel, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        d1 = self.relu(self.dilate1(x))
        d2 = self.relu(self.conv1x1(self.dilate2(x)))
        d3 = self.relu(self.conv1x1(self.dilate2(self.dilate1(x))))
        d4 = self.relu(self.conv1x1(self.dilate3(self.dilate2(self.dilate1(x)))))
        return x + d1 + d2 + d3 + d4


class RMPblock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.pool1 = nn.MaxPool2d(2, stride=2)
        self.pool2 = nn.MaxPool2d(3, stride=3)
        self.pool3 = nn.MaxPool2d(5, stride=5)
        self.pool4 = nn.MaxPool2d(6, stride=6)
        self.conv = nn.Conv2d(in_channels, 1, 1)

    def forward(self, x):
        size = x.shape[2:]
        p1 = F.interpolate(self.conv(self.pool1(x)), size=size, mode='bilinear', align_corners=True)
        p2 = F.interpolate(self.conv(self.pool2(x)), size=size, mode='bilinear', align_corners=True)
        p3 = F.interpolate(self.conv(self.pool3(x)), size=size, mode='bilinear', align_corners=True)
        p4 = F.interpolate(self.conv(self.pool4(x)), size=size, mode='bilinear', align_corners=True)
        return torch.cat([x, p1, p2, p3, p4], dim=1)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels // 4, 1)
        self.bn1 = nn.BatchNorm2d(in_channels // 4)
        self.deconv = nn.ConvTranspose2d(in_channels // 4, in_channels // 4, 3, stride=2, padding=1, output_padding=1)
        self.bn2 = nn.BatchNorm2d(in_channels // 4)
        self.conv2 = nn.Conv2d(in_channels // 4, out_channels, 1)
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.deconv(x)))
        x = self.relu(self.bn3(self.conv2(x)))
        return x


class CENet(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        resnet = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)

        self.firstconv = resnet.conv1
        self.firstbn = resnet.bn1
        self.firstrelu = resnet.relu
        self.firstmaxpool = resnet.maxpool

        self.encoder1 = resnet.layer1
        self.encoder2 = resnet.layer2
        self.encoder3 = resnet.layer3
        self.encoder4 = resnet.layer4

        self.dac = DACblock(512)
        self.rmp = RMPblock(512)

        self.decoder4 = DecoderBlock(516, 256)
        self.decoder3 = DecoderBlock(512, 128)
        self.decoder2 = DecoderBlock(256, 64)
        self.decoder1 = DecoderBlock(128, 64)

        self.finaldeconv1 = nn.ConvTranspose2d(128, 32, 4, stride=2, padding=1)
        self.finalconv2 = nn.Conv2d(32, 32, 3, padding=1)
        self.finalconv3 = nn.Conv2d(32, num_classes, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        e0 = self.firstrelu(self.firstbn(self.firstconv(x)))
        x = self.firstmaxpool(e0)

        e1 = self.encoder1(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)

        dac_out = self.dac(e4)
        rmp_out = self.rmp(dac_out)

        d4 = self.decoder4(rmp_out)
        d4_cat = torch.cat([d4, e3], dim=1)

        d3 = self.decoder3(d4_cat)
        d3_cat = torch.cat([d3, e2], dim=1)

        d2 = self.decoder2(d3_cat)
        d2_cat = torch.cat([d2, e1], dim=1)

        d1 = self.decoder1(d2_cat)
        d1_cat = torch.cat([d1, e0], dim=1)

        out = self.relu(self.finaldeconv1(d1_cat))
        out = self.relu(self.finalconv2(out))
        out = self.finalconv3(out)

        return out

# ==========================================
# BIẾN THỂ
# ==========================================
class CENet_EffB4(nn.Module):
    """
    CENet with EfficientNet-B4 encoder (MBConv / depthwise-separable).
    Encoder channels: e0=24  e1=32  e2=56  e3=160  e4=448
    Best accuracy-to-speed ratio; ~19 M total params.
    """
    def __init__(self, num_classes=1, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained=pretrained,
            features_only=True,
            out_indices=(0, 1, 2, 3, 4),
        )
        # 448×448 verified:
        #   stage0: (B,  24, 224, 224)
        #   stage1: (B,  32, 112, 112)
        #   stage2: (B,  56,  56,  56)
        #   stage3: (B, 160,  28,  28)
        #   stage4: (B, 448,  14,  14)

        self.dac = DACblock(448)
        self.rmp = RMPblock(448)    # → 452ch

        self.decoder4 = DecoderBlock(452, 160)
        self.decoder3 = DecoderBlock(320, 56)   # 160 + 160 skip
        self.decoder2 = DecoderBlock(112, 32)   # 56  + 56  skip
        self.decoder1 = DecoderBlock(64,  24)   # 32  + 32  skip

        # Final upsample ×2 + ×2 back to full resolution
        self.finaldeconv1 = nn.ConvTranspose2d(48, 32, 4, stride=2, padding=1)
        self.finalconv2   = nn.Conv2d(32, 32, 3, padding=1)
        self.finalconv3   = nn.Conv2d(32, num_classes, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        e0, e1, e2, e3, e4 = self.backbone(x)

        rmp_out = self.rmp(self.dac(e4))                       # 452ch

        d4 = self.decoder4(rmp_out)                            # 160ch
        d3 = self.decoder3(torch.cat([d4, e3], 1))            # 56ch
        d2 = self.decoder2(torch.cat([d3, e2], 1))            # 32ch
        d1 = self.decoder1(torch.cat([d2, e1], 1))            # 24ch

        out = self.relu(self.finaldeconv1(torch.cat([d1, e0], 1)))  # 48→32, ×2
        out = self.relu(self.finalconv2(out))
        out = self.finalconv3(out)
        return out


class CENet_MobileV3(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'mobilenetv3_large_100',
            pretrained=pretrained,
            features_only=True,
            out_indices=(0, 1, 2, 3, 4),
        )
        # 448×448 verified:
        #   stage0: (B,  16, 224, 224)
        #   stage1: (B,  24, 112, 112)
        #   stage2: (B,  40,  56,  56)
        #   stage3: (B, 112,  28,  28)
        #   stage4: (B, 960,  14,  14)

        # Project 960 → 448 trước DAC (tránh DAC quá nặng)
        self.bottleneck_conv = nn.Conv2d(960, 448, 1)
        self.bottleneck_bn   = nn.BatchNorm2d(448)
        self.bottleneck_relu = nn.ReLU(inplace=True)

        self.dac = DACblock(448)
        self.rmp = RMPblock(448)              # → 452ch

        self.decoder4 = DecoderBlock(452, 112)
        self.decoder3 = DecoderBlock(224,  40)  # 112 + e3(112) = 224
        self.decoder2 = DecoderBlock( 80,  24)  #  40 + e2( 40) =  80
        self.decoder1 = DecoderBlock( 48,  16)  #  24 + e1( 24) =  48

        # cat(d1=16, e0=16) = 32ch → ×2 → full resolution
        self.finaldeconv1 = nn.ConvTranspose2d(32, 32, 4, stride=2, padding=1)
        self.finalconv2   = nn.Conv2d(32, 32, 3, padding=1)
        self.finalconv3   = nn.Conv2d(32, num_classes, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # Encoder
        e0, e1, e2, e3, e4 = self.backbone(x)
        #  16ch/2   24ch/4  40ch/8  112ch/16  960ch/32

        # Bottleneck projection: 960 → 448
        e4 = self.bottleneck_relu(self.bottleneck_bn(self.bottleneck_conv(e4)))

        # DAC + RMP
        dac_out = self.dac(e4)
        rmp_out = self.rmp(dac_out)  # 260ch

        # Decoder
        d4 = self.decoder4(rmp_out)                       # 112ch, /16
        d3 = self.decoder3(torch.cat([d4, e3], dim=1))   #  40ch, /8
        d2 = self.decoder2(torch.cat([d3, e2], dim=1))   #  24ch, /4
        d1 = self.decoder1(torch.cat([d2, e1], dim=1))   #  16ch, /2

        # Final head
        out = self.relu(self.finaldeconv1(torch.cat([d1, e0], dim=1)))  # 32ch, /1
        out = self.relu(self.finalconv2(out))
        out = self.finalconv3(out)
        return out

class CENet_SwinT(nn.Module):
    """
    CENet với Swin Transformer-Tiny encoder.

    Swin-T stage channels : e0=96  e1=192  e2=384  e3=768
    Spatial resolution    : e0=H/4  e1=H/8  e2=H/16  e3=H/32

    DecoderBlock tự upsample ×2 bên trong (ConvTranspose2d stride=2)
    nên sau 4 decoder: H/32 → H/16 → H/8 → H/4 → H/2
    Chỉ cần 1× finaldeconv để về H (giống CENet gốc).
    """
    def __init__(self, num_classes=1, img_size=512, pretrained=True):
        super().__init__()

        # ── Encoder ────────────────────────────────────────────────
        self.backbone = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=pretrained,
            features_only=True,
            img_size=img_size,
            out_indices=(0, 1, 2, 3),
        )
        ec = self.backbone.feature_info.channels()  # [96, 192, 384, 768]

        # ── Context Extractor ──────────────────────────────────────
        self.dac = DACblock(ec[3])        # 768 → 768
        self.rmp = RMPblock(ec[3])        # 768 → 772

        # ── Decoder ────────────────────────────────────────────────
        # Mỗi DecoderBlock tự upsample ×2 bên trong
        # decoder4: 772       → 384,  H/32 → H/16;  cat e2(384) → 768
        # decoder3: 768       → 192,  H/16 → H/8;   cat e1(192) → 384
        # decoder2: 384       → 96,   H/8  → H/4;   cat e0(96)  → 192
        # decoder1: 192       → 96,   H/4  → H/2    (không skip)
        self.decoder4 = DecoderBlock(ec[3] + 4, ec[2])   # 772 → 384
        self.decoder3 = DecoderBlock(ec[2] * 2, ec[1])   # 768 → 192
        self.decoder2 = DecoderBlock(ec[1] * 2, ec[0])   # 384 → 96
        self.decoder1 = DecoderBlock(ec[0] * 2, ec[0])   # 192 → 96

        # ── Final head ─────────────────────────────────────────────
        # d1 ra H/2 → 1× ConvTranspose2d → H (giống CENet gốc)
        self.finaldeconv1 = nn.ConvTranspose2d(ec[0], 32, 4, stride=2, padding=1)
        self.finalconv2   = nn.Conv2d(32, 32, 3, padding=1)
        self.finalconv3   = nn.Conv2d(32, num_classes, 1)
        self.relu         = nn.ReLU(inplace=True)

    @staticmethod
    def _bchw(feats):
        """Swin trả về (B,H,W,C) → (B,C,H,W)."""
        return [f.permute(0, 3, 1, 2).contiguous() for f in feats]

    def forward(self, x):
        # ── Encoder ────────────────────────────────────────────────
        e0, e1, e2, e3 = self._bchw(self.backbone(x))
        # e0: B×96 ×H/4 ×W/4
        # e1: B×192×H/8 ×W/8
        # e2: B×384×H/16×W/16
        # e3: B×768×H/32×W/32

        # ── Context Extractor ──────────────────────────────────────
        dac_out = self.dac(e3)
        rmp_out = self.rmp(dac_out)   # B×772×H/32×W/32

        # ── Decoder + skip ─────────────────────────────────────────
        d4     = self.decoder4(rmp_out)            # B×384×H/16
        d4_cat = torch.cat([d4, e2], dim=1)        # B×768×H/16

        d3     = self.decoder3(d4_cat)             # B×192×H/8
        d3_cat = torch.cat([d3, e1], dim=1)        # B×384×H/8

        d2     = self.decoder2(d3_cat)             # B×96×H/4
        d2_cat = torch.cat([d2, e0], dim=1)        # B×192×H/4

        d1     = self.decoder1(d2_cat)             # B×96×H/2

        # ── Final head ─────────────────────────────────────────────
        out = self.relu(self.finaldeconv1(d1))     # B×32×H
        out = self.relu(self.finalconv2(out))      # B×32×H
        return self.finalconv3(out)                # B×num_classes×H×W

In [ ]:
def load_model(dataset_name, model_key, loss_func, device):
    result_dir = os.path.join(RESULT_BASE, f'result_{dataset_name}')
    ckpt_path  = os.path.join(result_dir, f'best_model_{model_key}_{loss_func}.pth')

    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f'Không tìm thấy checkpoint:\n{ckpt_path}')

    if model_key == 'unet':
        model = UNet(n_channels=3, n_classes=1, bilinear=True)
    elif model_key == 'r34':
        model = CENet(num_classes=1)
    elif model_key == 'effb4':
        model = CENet_EffB4(num_classes=1, pretrained=False)
    elif model_key == 'mobilev3':
        model = CENet_MobileV3(num_classes=1, pretrained=False)
    elif model_key == 'swin':
        model = CENet_SwinT(num_classes=1, img_size = 448,pretrained=False)
    else:
        raise ValueError(f'Model không hợp lệ: {model_key}')

    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device).eval()

    saved_epoch   = checkpoint.get('epoch', '?')
    saved_metrics = checkpoint.get('metrics', {})
    print(f'  Loaded  : {os.path.basename(ckpt_path)}')
    print(f'  Best epoch: {saved_epoch}')
    if saved_metrics:
        print(f'  Metrics  : Dice={saved_metrics.get("Dice",0):.4f} | '
              f'E={saved_metrics.get("E",0):.4f} | '
              f'Sen={saved_metrics.get("Sen",0):.4f} | '
              f'Acc={saved_metrics.get("Acc",0):.4f}')
    return model

In [ ]:
def preprocess(image_path, dataset_name, img_size=448):
    cfg = DATASET_CONFIGS[dataset_name]
    img = Image.open(image_path)
    img = img.convert('RGB') if cfg['is_rgb'] else img.convert('L').convert('RGB')

    tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=cfg['mean'], std=cfg['std']),
    ])
    return tf(img).unsqueeze(0)  # (1, 3, H, W)

In [ ]:
@torch.no_grad()
def predict(model, tensor, device, threshold=0.5):
    tensor = tensor.to(device)
    logits = model(tensor)
    prob   = torch.sigmoid(logits)
    mask   = (prob > threshold).float()
    return prob.squeeze().cpu().numpy(), mask.squeeze().cpu().numpy()

def mask_stats(mask):
    total_px    = mask.size
    lung_px     = int(mask.sum())
    lung_ratio  = lung_px / total_px * 100
    return {'total_px': total_px, 'lung_px': lung_px, 'lung_ratio': lung_ratio}

def visualize(image_path, prob_map, mask, dataset_name, model_label, loss_func, threshold=0.5):
    orig = Image.open(image_path).convert('RGB')
    orig_np = np.array(orig)

    mask_resized = np.array(
        Image.fromarray((mask * 255).astype(np.uint8)).resize(
            (orig_np.shape[1], orig_np.shape[0]), Image.NEAREST
        )
    ) / 255.0

    prob_resized = np.array(
        Image.fromarray((prob_map * 255).astype(np.uint8)).resize(
            (orig_np.shape[1], orig_np.shape[0]), Image.BILINEAR
        )
    ) / 255.0

    stats = mask_stats(mask_resized)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    fig.patch.set_facecolor('#1a1a2e')
    title_kw = dict(fontsize=11, color='white', pad=8)

    # 1. Ảnh gốc
    axes[0].imshow(orig_np, cmap='gray' if not DATASET_CONFIGS[dataset_name]['is_rgb'] else None)
    axes[0].set_title('Ảnh gốc', **title_kw)

    # 2. Probability map
    im = axes[1].imshow(prob_resized, cmap='plasma', vmin=0, vmax=1)
    axes[1].set_title('Probability map', **title_kw)
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    # 3. Overlay
    overlay = orig_np.copy().astype(float)
    green_mask = mask_resized > 0.5
    overlay[green_mask, 0] = overlay[green_mask, 0] * 0.4
    overlay[green_mask, 1] = overlay[green_mask, 1] * 0.4 + 255 * 0.6
    overlay[green_mask, 2] = overlay[green_mask, 2] * 0.4
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    axes[2].imshow(overlay)
    axes[2].contour(mask_resized, levels=[0.5], colors=['#00ffcc'], linewidths=1.2)
    axes[2].set_title('Overlay', **title_kw)

    for ax in axes:
        ax.axis('off')

    suptitle = (f'Dataset: {dataset_name}  |  Model: {model_label}  |  Loss: {loss_func}  |  '
                f'Phổi: {stats["lung_px"]:,}px ({stats["lung_ratio"]:.1f}%)')
    fig.suptitle(suptitle, fontsize=11, color='#aaaacc', y=1.01)

    plt.tight_layout()
    plt.show()
    print(f'  Tổng pixel: {stats["total_px"]:,} | Pixel phổi: {stats["lung_px"]:,} | Tỉ lệ: {stats["lung_ratio"]:.2f}%')

In [ ]:
def run_demo():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}\n')

    w_dataset = widgets.Dropdown(options=list(DATASET_CONFIGS.keys()), description='Dataset:')
    w_model   = widgets.Dropdown(options=list(MODEL_OPTIONS.keys()),   description='Model:')
    w_loss    = widgets.Dropdown(options=LOSS_OPTIONS,                 description='Loss:')
    w_upload  = widgets.FileUpload(accept='image/*', multiple=False,   description='Chọn ảnh')
    w_btn     = widgets.Button(description='▶ Chạy phân đoạn',
                               button_style='primary', layout=widgets.Layout(width='200px'))
    w_out     = widgets.Output()

    ui = widgets.VBox([
        widgets.HTML('<h3 style="color:#4a90d9">🫁 Medical Image Segmentation Demo</h3>'),
        widgets.HBox([w_dataset, w_model, w_loss]),
        w_upload,
        w_btn,
        w_out,
    ])
    display(ui)

    def on_run(b):
        w_out.clear_output()
        with w_out:
            if not w_upload.value:
                print('⚠ Vui lòng upload ảnh trước!'); return

            fname = list(w_upload.value.keys())[0]
            fdata = w_upload.value[fname]['content']
            tmp   = f'/tmp/{fname}'
            with open(tmp, 'wb') as f:
                f.write(fdata)

            dataset_name = w_dataset.value
            model_label  = w_model.value
            model_key    = MODEL_OPTIONS[model_label]
            loss_func    = w_loss.value

            try:
                print('Loading model...')
                model = load_model(dataset_name, model_key, loss_func, device)
                tensor = preprocess(tmp, dataset_name)
                prob, mask = predict(model, tensor, device, threshold=0.5)
                visualize(tmp, prob, mask, dataset_name, model_label, loss_func, threshold=0.5)
            except FileNotFoundError as e:
                print(f'❌ {e}')
            except Exception as e:
                import traceback; traceback.print_exc()

    w_btn.on_click(on_run)

In [ ]:
run_demo()

Device: cuda

